# 02 — Tiền xử lý dữ liệu (Data Preprocessing)

This notebook prepares data for EDA: it loads project datasets, applies cleaning and transformations, and saves preprocessed outputs for downstream analysis.


In [ ]:
# Import thư viện và hàm trợ giúp
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import json

# Thêm thư mục gốc của project vào sys.path để có thể import `src` khi chạy notebook từ cwd khác
proj_root = Path('.').resolve()
if str(proj_root) not in sys.path:
    sys.path.append(str(proj_root))

# Import các hàm tiền xử lý nội bộ
from src.text_preprocess import process_dataframe, clean_data, process_text

print('pandas:', pd.__version__)
print('numpy:', np.__version__)


In [ ]:
# Cấu hình và seed
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path('data/processed')
DATA_IN_PRIMARY = DATA_DIR / 'combined_labeled.csv'
DATA_IN_FALLBACK = DATA_DIR / 'combined_balanced.csv'
DATA_OUT = DATA_DIR / 'combined_preprocessed.csv'
SCALER_OUT = Path('models') / 'preprocessing_scaler.joblib'
METADATA_OUT = DATA_DIR / 'preprocessing_metadata.json'

print('Primary input:', DATA_IN_PRIMARY)
print('Fallback input:', DATA_IN_FALLBACK)
print('Output:', DATA_OUT)


In [ ]:
# Nạp dữ liệu (chọn file chính nếu có)
input_path = DATA_IN_PRIMARY if DATA_IN_PRIMARY.exists() else DATA_IN_FALLBACK if DATA_IN_FALLBACK.exists() else None
if input_path is None:
    raise FileNotFoundError('Không tìm thấy CSV đầu vào trong data/processed. Vui lòng chạy script tiền xử lý hoặc đặt file CSV vào.')

print('Loading:', input_path)
df = pd.read_csv(input_path)
print('Shape:', df.shape)

df.head()


In [ ]:
# Kiểm tra dữ liệu nhanh
print('Columns:', df.columns.tolist())
print('\nInfo:')
df.info()

# Đếm giá trị cột `label` nếu tồn tại
if 'label' in df.columns:
    print('\nPhân phối nhãn:')
    display(df['label'].value_counts(normalize=True))

# Tổng hợp giá trị thiếu
missing = df.isna().mean().sort_values(ascending=False)
print('\nTỷ lệ missing theo cột:')
print(missing[missing>0])


In [ ]:
# Xử lý giá trị thiếu và bản ghi trùng lặp
# Chiến lược ví dụ: loại bỏ hàng thiếu text và duplicate
text_col_candidates = [c for c in df.columns if c.lower()=='text' or c.lower().endswith('_text')]
text_col = text_col_candidates[0] if text_col_candidates else 'text'

if text_col not in df.columns:
    raise ValueError(f'Không tìm thấy cột text; mong đợi {text_col}')

before_shape = df.shape

df = df.drop_duplicates().reset_index(drop=True)
df = df[df[text_col].notna()].reset_index(drop=True)

print('Before:', before_shape, 'After drop dup & NA:', df.shape)


In [ ]:
# Áp dụng tiền xử lý văn bản từ `src.text_preprocess`
print('Applying text preprocessing to', text_col)
processed = process_dataframe(df, text_column=text_col, output_column='clean_text', save_path=None, min_text_length=0)

# Hiển thị mẫu: gốc vs đã làm sạch
processed[[text_col, 'clean_text']].head(10)


In [ ]:
# Lưu CSV đã xử lý và metadata
DATA_OUT.parent.mkdir(parents=True, exist_ok=True)
processed.to_csv(DATA_OUT, index=False, encoding='utf-8')

metadata = {
    'input_path': str(input_path),
    'output_path': str(DATA_OUT),
    'n_rows_input': int(df.shape[0]),
    'n_rows_output': int(processed.shape[0]),
    'text_column': text_col,
}
with open(METADATA_OUT, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print('Đã lưu CSV đã xử lý tại', DATA_OUT)
print('Đã lưu metadata tại', METADATA_OUT)


In [ ]:
# Kiểm tra nhanh và test
assert processed.shape[0] > 0, 'Processed dataframe is empty'
assert 'clean_text' in processed.columns
assert processed['clean_text'].isna().sum() == 0

print('Kiểm tra sanity passed. Số hàng sau xử lý:', processed.shape[0])

# Ghi chú bước tiếp theo (link tới EDA)
print('\nBước tiếp theo: mở notebook EDA tại notebooks/01_eda.ipynb và sử dụng file đã lưu:', DATA_OUT)
